# 核心复习笔记：CNN 工程细节与贝叶斯推断

## 一、 张量维度心算 (万能公式)
这是 debug 卷积网络最核心的武器。假设输入图像为正方形，输出单边尺寸公式为：

$$H_{out} = \lfloor (H_{in} + 2p - k)/s \rfloor + 1$$
*(注：$H_{in}$ 为输入尺寸，$p$ 为单侧填充，$k$ 为卷积核大小，$s$ 为步幅。计算时注意向下取整。)*

**三大经典网络层的尺寸直觉：**
* **VGG 保尺寸层** ($k=3, p=1, s=1$)：进出尺寸完全一致。
* **暴力降采样层** ($k=3, p=1, s=2$)：特征图长宽直接精准腰斩。
* **1x1 隐身层** ($k=1, p=0, s=1$)：特征图空间尺寸不变，只改变通道数。

---

## 二、 CNN 核心组件的物理直觉 (d2l 6.3 - 6.5)

### 1. 填充 (Padding) 与 步幅 (Stride)
* **填充 (Padding)**：用来保护边缘信息，防止深层网络把图片“卷”没，并人为控制输出特征图的大小。
* **步幅 (Stride)**：降维打击的利器。通过跳跃滑动大幅减少计算量，同时快速扩大下一层的感受野。

### 2. 多通道 (Multi-Channels)
* **输入通道**：面对 RGB 图像，卷积核变成 3D 砖块（如 $3 \times 3 \times 3$），在空间上滑动，在通道上同时相乘求和。
* **输出通道**：代表你雇佣了多少个不同的“特征提取器”。输出通道数越多，网络能抓取的特征种类越丰富。
* **核心考点：1x1 卷积**：本质是一个**跨通道的像素级全连接层 (MLP)**。它不提取空间局部特征，专门用来做通道维度的降维或升维（融合特征），极大地节省参数量。

### 3. 池化层 (Pooling)
* 通常指最大池化 (Max Pooling)。不需要学习参数，是一个粗暴的降采样操作。
* **核心作用**：保留局部最强特征，赋予网络极强的**抗空间平移抖动能力**（只要特征在池化窗口内，选出的最大值就不变），同时暴力降维加速网络。

---

## 三、 感受野 (Receptive Field)
* **定义**：深层特征图上的某一个像素点，所融合的原始输入图像上的面积大小。
* **叠加效应**：两个 $3 \times 3$ 卷积层堆叠，能达到一个 $5 \times 5$ 卷积层的感受野，但参数量大幅减少（$18 C^2$ vs $25 C^2$），且增加了非线性表达能力。
* **结论**：网络越深、经过的池化层越多，高层神经元的感受野就越大，从而能理解更宏观的语义特征。

---

## 四、 贝叶斯推断核心思路
贝叶斯定理是机器学习“根据新数据更新认知”的数学灵魂。

* **核心公式**：$$P(H|D) = \frac{P(D|H) P(H)}{P(D)}$$
* **终极口诀**：**后验 $\propto$ 似然 $\times$ 先验**

**实战推断的两大直觉：**
1. **先验的碾压力量**：当基础发病率（先验）极低时，即便试剂准确率（似然）高达 99%，单次阳性结果（后验）真正患病的概率依然极低。
2. **滚动的认知更新**：在连续观测中，前一次算出的结果（后验）会直接变成下一次观测的起点（新先验）。多次叠加相同结果，会迅速放大似然的权重，彻底翻转初始认知。